# OpenPlaque — LCX Distal Source-CCTA Reacquisition (fixed)
Blind source-space distal extension of C6/C7, followed only afterward by validated chamber-interface scoring. Single-kernel execution; no Python subprocess. Includes Dijkstra coordinate-index bugfix only; scientific thresholds are unchanged.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/OpenPlaque'
OUTPUT_DIR = DRIVE_ROOT + '/LCX_Distal_Reacquisition_v1_fixed'
BRANCH = 'lcx-distal-reacquisition-from-main'
BASELINE = '0593b453959f5a353d644267fbeef24b514ef4d7'
print('Branch:', BRANCH)
print('Output:', OUTPUT_DIR)


In [ ]:
import os, shutil
repo='/content/OpenPlaque'
if os.path.exists(repo): shutil.rmtree(repo)
!git clone --depth 1 --branch $BRANCH https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
HEAD = !git -C /content/OpenPlaque rev-parse HEAD
HEAD = HEAD[0].strip()
print('Checked out HEAD:', HEAD)


In [ ]:
%pip uninstall -y openplaque >/dev/null 2>&1
%pip install -q --no-cache-dir --force-reinstall --no-deps /content/OpenPlaque
%pip install -q pytest SimpleITK scipy pandas matplotlib numpy


In [ ]:
import sys, importlib, inspect, pytest
for name in list(sys.modules):
    if name == 'openplaque' or name.startswith('openplaque.'):
        del sys.modules[name]
importlib.invalidate_caches()
import openplaque
from openplaque import lcx_distal_reacquisition_fixed as exp
print('openplaque:', openplaque.__file__)
print('algorithm:', exp.ALGORITHM)
print('patch:', exp.PATCH_VERSION)
assert exp.BASELINE == BASELINE
assert exp.ALGORITHM == 'lcx-distal-reacquisition-v1.0-lowmem'
assert exp.PATCH_VERSION == 'dijkstra-distance-shape-v1'
print('self-test:', exp.synthetic_reacquisition_self_test())
rc = pytest.main(['-q','/content/OpenPlaque/tests/test_lcx_distal_reacquisition.py','/content/OpenPlaque/tests/test_lcx_distal_reacquisition_fixed.py'])
if rc != 0: raise RuntimeError(f'pytest failed with exit code {rc}')


In [ ]:
from pathlib import Path
root=Path(DRIVE_ROOT)
required=[
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
 root/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
 root/'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',
 root/'PCAT_RCA_10_50/rca_centerline_smoothed_zyx.csv',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries_LEGACY/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_atrium_left.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_ventricle_left.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_atrium_right.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_ventricle_right.nii.gz',
 root/'Joint_Three_Vessel_Template_Classifier_v1/LCX_joint_candidate_ranking.csv',
]+[root/f'Joint_Three_Vessel_Template_Classifier_v1/candidate_{i:02d}_source_path.csv' for i in range(1,6)]
missing=[str(p) for p in required if not p.exists()]
print('Preflight required:',len(required),'missing:',len(missing))
if missing: raise FileNotFoundError('\n'.join(missing))


In [ ]:
import gc
from openplaque.lcx_distal_reacquisition_fixed import run
gc.collect()
result = run(DRIVE_ROOT, OUTPUT_DIR)
print('STATUS:', result['summary']['status'])
print('SOURCE REACQUISITION:', result['summary']['source_reacquisition'])
print('CHAMBER CONTROLS:', result['summary']['chamber_interface_controls'])
print('DECISION:', result['summary']['decision'])
print('REPORT:', result['report'])
print('ZIP:', result['zip'])
